<a href="https://colab.research.google.com/github/alimovscott/cloneGPT/blob/main/clone_gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [33]:
!pip install transformers torch bitsandbytes datasets peft trl

In [ ]:
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM, TrainingArguments
import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer

In [ ]:
model_id = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
tokenizer = AutoTokenizer.from_pretrained(model_id)

# print("Vocab size:", tokenizer.vocab_size)
# print('Special tokens:', tokenizer.special_tokens_map)


# quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16
)

bnb_config

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map='auto',
    # dtype=torch.bfloat16

    )
#

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [29]:
# Before Fine-tuning
prompt = "Explain what a tokenizer is? "
# promt = "A tokenizer is a tool in natural language processing that"

inputs = tokenizer(
    prompt,
    return_tensors='pt'
).to(model.device)

with torch.no_grad():
  output_ids = model.generate(
      **inputs,
      max_new_tokens=80,
      do_sample=True,
      temperature=0.7
  )

  print(tokenizer.decode(output_ids[0], skip_special_tokens=True))

[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Explain what a tokenizer is? 

In Python, a **Tokenizer** is a class used to tokenize a text into a list of tokens. This is done by breaking the text into smaller segments and then comparing each segment to a set of pre-defined tokens. The pre-defined tokens are usually defined in a separate file called a tokenizer.py, and the tokenizer class is then used to return a list


In [ ]:
def count_parameters(model):
  return sum(p.numel() for p in model.parameters() )

total_params = count_parameters(model)
print(f'Total parameters: {total_params:,}')



Total parameters: 615,606,272


In [ ]:
## datasets
## instraction tuning

In [ ]:

dataset = load_dataset("yahma/alpaca-cleaned", split="train")
dataset[0]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

alpaca_data_cleaned.json: reconstructing file:   0%|          |  0.00B / 44.3MB            

alpaca_data_cleaned.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/51760 [00:00<?, ? examples/s]

{'output': '1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.\n\n2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.\n\n3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.',
 'input': '',
 'instruction': 'Give three tips for staying healthy.'}

In [ ]:
def generate_prompt(example):
  instruction = example['instruction']
  input_text = example['input']
  output_text = example['output']

  if input_text:
    return(
    "### Instruction:\n"
    f"{instruction}\n\n"
    "### Input:\n\n"
    f"{input_text}\n\n"
    "### Response:\n"
    f"{output_text}"
    )
  else:
    return(
        "### Instruction:\n"
        f"{instruction}\n\n"
        "### Response:\n"
        f"{output_text}"
    )


# generate_prompt(dataset[1])


def formatting_func(example):
  return{'text': generate_prompt(example)}

dataset = dataset.map(formatting_func)




Map:   0%|          | 0/51760 [00:00<?, ? examples/s]

In [ ]:
dataset[0]['text']

'### Instruction:\nGive three tips for staying healthy.\n\n### Response:\n1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.\n\n2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.\n\n3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.'

In [ ]:
dataset = dataset.select(range(7000))

In [ ]:
dataset = dataset.shuffle(seed=42)

In [ ]:
# Full Fine- tuning =>
# cheap Fine-tuning =>
# PEFT => paremetr Efficent Fine Tuning
# OOM => Out of Memory

In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"],
)




In [ ]:
model = get_peft_model(model, lora_config)


In [ ]:
model.print_trainable_parameters()

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


In [ ]:
# QLora
# Lora

In [ ]:
training_args = TrainingArguments(
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    logging_steps=20,
    output_dir="./clone_gpt",
    save_strategy="epoch",
    bf16=True,
    fp16=False,
    report_to="none"

)

In [ ]:
print(dataset.column_names)

['output', 'input', 'instruction', 'text']


In [20]:
#SFTTrainer vs Trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    formatting_func=lambda x: x['text'],
    args=training_args
    )

trainer.train()
model.save_pretrained('clone_gpt')
tokenizer.save_pretrained('clone_gpt')

Applying formatting function to train dataset:   0%|          | 0/7000 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/7000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/7000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/7000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/7000 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/7000 [00:00<?, ? examples/s]

Step,Training Loss
20,1.534991
40,1.587553
60,1.322445
80,1.373602
100,1.304839
120,1.321419
140,1.349374
160,1.244505
180,1.307150
200,1.215225


Step,Training Loss
20,1.534991
40,1.587553
60,1.322445
80,1.373602
100,1.304839
120,1.321419
140,1.349374
160,1.244505
180,1.307150
200,1.215225


('clone_gpt/tokenizer_config.json',
 'clone_gpt/chat_template.jinja',
 'clone_gpt/tokenizer.json')

In [23]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch


base_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(base_model_id)
model_base = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    device_map='auto',

)

model_base.eval()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (up_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (down_proj): Linear(in_features=5632, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (rot

In [28]:
prompt = "Explain what machine learning is in simple words."

inputs_base = tokenizer(prompt, return_tensors="pt").to(model_base.device)

with torch.no_grad():
  output_base = model_base.generate(
      **inputs_base,
      max_new_tokens=120,
      temperature=0.7,
      do_sample=True,
  )


print("==== BASE MODEL OUTPUT ====")
print(tokenizer.decode(output_base[0], skip_special_tokens=True))


[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


==== BASE MODEL OUTPUT ====
Explain what machine learning is in simple words. What is machine learning and why is it important? How does it differ from traditional programming techniques? Provide examples of real-world applications of machine learning.


#FINE_TUNED MODEL INFERENCE

In [34]:
model_path = "clone_gpt"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [41]:
prompt = """### Instruction:
Explain what machine learning is in simple words.

### Response:
"""

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=120,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

print("==== FINE-TUNED MODEL OUTPUT ====")
print(tokenizer.decode(output[0], skip_special_tokens=True))

[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


==== FINE-TUNED MODEL OUTPUT ====
### Instruction:
Explain what machine learning is in simple words.

### Response:
Machine learning is a branch of artificial intelligence that involves the collection, organization, and improvement of data in order to teach computers to make informed decisions based on that data. Machine learning uses algorithms or algorithms to analyze large amounts of data and create a model that can make predictions or make decisions based on that data. This model can then be used to make predictions or make decisions in many areas, such as finance, healthcare, and customer service.


In [42]:
!zip -r clone_gpt.zip clone_gpt

  adding: clone_gpt/ (stored 0%)
  adding: clone_gpt/checkpoint-3500/ (stored 0%)
  adding: clone_gpt/checkpoint-3500/training_args.bin (deflated 53%)
  adding: clone_gpt/checkpoint-3500/adapter_config.json (deflated 59%)
  adding: clone_gpt/checkpoint-3500/README.md (deflated 66%)
  adding: clone_gpt/checkpoint-3500/adapter_model.safetensors (deflated 23%)
  adding: clone_gpt/checkpoint-3500/optimizer.pt (deflated 22%)
  adding: clone_gpt/checkpoint-3500/scheduler.pt (deflated 61%)
  adding: clone_gpt/checkpoint-3500/trainer_state.json (deflated 81%)
  adding: clone_gpt/checkpoint-3500/tokenizer_config.json (deflated 46%)
  adding: clone_gpt/checkpoint-3500/tokenizer.json (deflated 85%)
  adding: clone_gpt/checkpoint-3500/rng_state.pth (deflated 26%)
  adding: clone_gpt/checkpoint-3500/chat_template.jinja (deflated 60%)
  adding: clone_gpt/adapter_config.json (deflated 59%)
  adding: clone_gpt/README.md (deflated 44%)
  adding: clone_gpt/adapter_model.safetensors (deflated 23%)
  addi

In [43]:
from google.colab import files
files.download("clone_gpt.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [44]:
#####################
#   MODEL TESTING.  #
#####################